In [4]:
# ============================================================
# 1. Setup & Dataset Download
# ============================================================
import os, json, cv2,random,shutil
import numpy as np
from pathlib import Path
from PIL import Image
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
import torchvision as tv
import torchvision.transforms as T
import gradio as gr

ModuleNotFoundError: No module named 'torch'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

In [ ]:
dataset_path = Path("/content/drive/MyDrive/PlantVillage")
print("Dataset path:", dataset_path)

# Find class folders (Tomato_, Potato_, Pepper_)
def find_class_roots(base: Path):
    roots = {}
    for root, dirs, files in os.walk(base):
        for d in dirs:
            if any(d.startswith(prefix) for prefix in ["Tomato_", "Potato_", "Pepper_", "Corn_", "Rice_"]):
                roots[d] = Path(root) / d
    return roots

class_dirs = find_class_roots(dataset_path)
if not class_dirs:
    raise RuntimeError(f"No class folders found under {dataset_path}.")
target_classes = list(class_dirs.keys())
print("Found classes:", target_classes)

In [ ]:
#Train/val split
work_root = Path("/content/data")
train_dir, val_dir = work_root/"train", work_root/"val"
train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)

def copy_split(src_class_dir: Path, dst_train: Path, dst_val: Path, split=0.8, seed=42):
    imgs = [p for p in src_class_dir.iterdir() if p.suffix.lower() in [".jpg",".jpeg",".png",".bmp",".tif",".tiff"]]
    random.Random(seed).shuffle(imgs)
    cut = int(split * len(imgs))
    for p in imgs[:cut]:
        (dst_train/src_class_dir.name).mkdir(parents=True, exist_ok=True)
        shutil.copy(p, dst_train/src_class_dir.name/p.name)
    for p in imgs[cut:]:
        (dst_val/src_class_dir.name).mkdir(parents=True, exist_ok=True)
        shutil.copy(p, dst_val/src_class_dir.name/p.name)

for cls in target_classes:
    copy_split(class_dirs[cls], train_dir, val_dir)

print("Prepared split at:", work_root)

In [ ]:
# 2. Data Preparation
# ============================================================
img_size = 224
tfm_train = tv.transforms.Compose([
    tv.transforms.Resize(256),
    tv.transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0)),
    tv.transforms.RandomHorizontalFlip(),
    tv.transforms.ColorJitter(0.15, 0.15, 0.1, 0.02),
    tv.transforms.ToTensor(),
    tv.transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tfm_val = tv.transforms.Compose([
    tv.transforms.Resize(256),
    tv.transforms.CenterCrop(img_size),
    tv.transforms.ToTensor(),
    tv.transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = tv.datasets.ImageFolder(str(train_dir), transform=tfm_train)
val_ds   = tv.datasets.ImageFolder(str(val_dir),   transform=tfm_val)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print("Class order:", train_ds.classes)

In [ ]:
# 3. Model Setup (ResNet-50)
# ============================================================
model = tv.models.resnet50(weights=tv.models.ResNet50_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(train_ds.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

In [ ]:
# ============================================================
# 4. Training Loop
# ============================================================
def evaluate(m, loader):
    m.eval()
    correct = total = 0
    with torch.inference_mode():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = m(xb).argmax(1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    return correct / max(1, total)

EPOCHS = 20  # ⬅️ adjust to 20–30 for ~98% accuracy
best_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    scheduler.step()

    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - val_acc={val_acc:.4f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "/content/resnet50_leaf_V2.pt")
        with open("/content/classes.txt", "w") as f:
            f.write("\n".join(train_ds.classes))
        print("✅ Saved new best (val_acc=%.4f)" % val_acc)

print("Training complete. Best val_acc=%.4f" % best_acc)

In [ ]:
def is_leaf_image(pil_img, green_threshold=0.05):
    """Quick check if image looks like a leaf by % of green pixels."""
    img = np.array(pil_img.convert("RGB"))
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    lower_green = np.array([25, 40, 20])
    upper_green = np.array([95, 255, 255])
    mask_green = cv2.inRange(hsv, lower_green, upper_green)
    green_ratio = (mask_green > 0).sum() / mask_green.size
    return green_ratio > green_threshold

In [ ]:
# 5) Severity estimator
def estimate_severity_pil(pil_img):
    img = np.array(pil_img.convert("RGB"))
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    lower_green = np.array([25, 40, 20])
    upper_green = np.array([95, 255, 255])
    mask_green = cv2.inRange(hsv, lower_green, upper_green)
    total = mask_green.size
    green_pixels = (mask_green > 0).sum()
    non_green_ratio = 1.0 - (green_pixels / max(1, total))
    sev = float(np.clip(non_green_ratio, 0, 1))
    if sev < 0.2:
        stage = "Early"
    elif sev < 0.5:
        stage = "Mid"
    else:
        stage = "Late"
    return sev, stage

In [ ]:
# ---------- 3) Advice map (with Other Crop fallback) ----------
advice_map = {
    "Tomato_healthy": {
        "summary": "Plant healthy ✅",
        "actions": ["Maintain good irrigation."],
        "prevention": ["Rotate crops."]
    },
    "Tomato_Late_blight": {
        "summary": "Fungal disease.",
        "actions": ["Remove infected leaves.", "Use Mancozeb."],
        "prevention": ["Avoid overhead watering."]
    },
    "Tomato_Early_blight": {
        "summary": "Dark concentric spots.",
        "actions": ["Use fungicides with chlorothalonil."],
        "prevention": ["Rotate crops."]
    },
    "Tomato_Septoria_leaf_spot": {
        "summary": "Small dark spots.",
        "actions": ["Remove leaves.", "Apply copper fungicides."],
        "prevention": ["Keep leaves dry."]
    },
    "Tomato_Tomato_mosaic_virus": {
        "summary": "Viral disease.",
        "actions": ["Destroy infected plants."],
        "prevention": ["Disinfect tools."]
    },
    "Potato_Early_blight": {
        "summary": "Brown spots with rings.",
        "actions": ["Apply Mancozeb."],
        "prevention": ["Field sanitation."]
    },
    "Potato_Late_blight": {
        "summary": "Dark lesions.",
        "actions": ["Destroy plants.", "Apply fungicides."],
        "prevention": ["Avoid wet conditions."]
    },
    "Potato_healthy": {
        "summary": "Plant healthy ✅",
        "actions": ["No treatment needed."],
        "prevention": ["Good soil management."]
    },
    "Pepper_bell_Bacterial_spot": {
        "summary": "Bacterial leaf spots.",
        "actions": ["Use copper bactericides."],
        "prevention": ["Rotate crops."]
    },
    "Pepper_bell_healthy": {
        "summary": "Plant healthy ✅",
        "actions": ["Standard care."],
        "prevention": ["Monitor regularly."]
    },
    "Corn_Blight": {
        "summary": "Fungal disease causing large oval or irregular gray-green lesions on leaves.",
        "actions": ["Apply fungicides containing Mancozeb or Chlorothalonil at early symptoms."],
        "prevention": ["Rotate crops with non-host plants.", "Plant resistant corn varieties."]
    },
    "Corn_Common_Rust": {
        "summary": "Rust fungus producing reddish-brown pustules on upper and lower leaf surfaces.",
        "actions": ["Scout regularly and treat at early infection stages."],
        "prevention": ["Plant early to avoid peak rust pressure.", "Improve field airflow with proper spacing."]
    },
    "Corn_Gray_Leaf_Spot": {
        "summary": "Gray rectangular lesions that merge, reducing photosynthesis and yield.",
        "actions": ["Remove infected residue after harvest."],
        "prevention": ["Practice crop rotation.", "Avoid excessive nitrogen which promotes lush foliage."]
    },
    "Corn_Healthy": {
        "summary": "Corn crop is healthy.",
        "actions": [
            "Continue recommended irrigation and fertilization schedule.",
            "Scout regularly to detect diseases early."
        ],
        "prevention": ["Use certified disease-free seeds.", "Maintain good field hygiene."]
    },
    "Rice_Bacterial_leaf_blight": {
        "summary": "Bacterial disease causing yellowing and wilting of leaves from tip downwards.",
        "actions": ["Apply copper-based bactericides.", "Avoid injuring plants during cultivation."],
        "prevention": ["Use resistant rice varieties.", "Ensure proper water management to reduce disease spread."]
    },
    "Rice_Brown_spot": {
        "summary": "Fungal disease causing brown circular spots with gray centers on leaves.",
        "actions": ["Improve field drainage to avoid waterlogging."],
        "prevention": ["Plant resistant rice varieties.", "Maintain proper plant spacing for air circulation."]
    },
    "Rice_Leaf smut": {
        "summary": "Fungal disease causing black sooty spots on leaves, reducing photosynthesis.",
        "actions": ["Remove severely infected plants to minimize spread."],
        "prevention": ["Use clean and disease-free seeds.", "Maintain proper nutrition to boost plant resistance."]
    },
    "Other_Crop": {
        "summary": "The model detected a leaf, but not Tomato/Potato/Pepper/Rice/Corn.",
        "actions": ["Currently, only Tomato, Potato, Corn, Rice and Pepper are supported."],
        "prevention": ["Try uploading a supported crop leaf."]
    }
}
with open("/content/advice.json", "w") as f:
    json.dump(advice_map, f, indent=2)

In [ ]:
# 6) Inference helpers
def load_model(model_path="/content/resnet50_leaf_V2.pt", classes_path="/content/classes.txt"):
    classes = Path(classes_path).read_text().splitlines()
    m = tv.models.resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, len(classes))
    m.load_state_dict(torch.load(model_path, map_location=device), strict=False)
    m.eval().to(device)
    return m, classes

infer_tfm = tv.transforms.Compose([
    tv.transforms.Resize(256),
    tv.transforms.CenterCrop(224),
    tv.transforms.ToTensor(),
    tv.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

MODEL, CLASSES = load_model()
ADVICE = json.loads(Path("/content/advice.json").read_text())


In [ ]:
# 🔑 Normalize class label for advice lookup
def normalize_label(label: str):
    return (
        label.replace("___", "_")
             .replace("__", "_")
    )


@torch.inference_mode()
def predict_image(pil_img: Image.Image):
    # Step 1: Leaf check
    if not is_leaf_image(pil_img):
        return {
            "label": "Invalid Input",
            "confidence": 0.0,
            "severity_score": 0.0,
            "severity_stage": "N/A",
            "advice": {
                "summary": "This does not appear to be a leaf image.",
                "actions": ["Please upload a clear photo of a crop leaf."],
                "prevention": []
            }
        }

    # Step 2: Model inference
    x = infer_tfm(pil_img).unsqueeze(0).to(device)
    logits = MODEL(x)
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    idx = int(np.argmax(probs))
    label, conf = CLASSES[idx], float(probs[idx])

    # Step 3: Confidence check
    if conf < 0.7:
        return {
            "label": "Uncertain Prediction",
            "confidence": conf,
            "severity_score": 0.0,
            "severity_stage": "N/A",
            "advice": {
                "summary": "The model is not confident this is a valid leaf image.",
                "actions": ["Upload a clearer photo of the leaf."],
                "prevention": []
            }
        }

    # Step 4: Severity
    sev_score, stage = estimate_severity_pil(pil_img)

    # Step 5: Advice with normalization
    norm_label = normalize_label(label)
    rec = ADVICE.get(norm_label, ADVICE.get(label, ADVICE["Other_Crop"]))

    return {
        "label": norm_label,
        "confidence": conf,
        "severity_score": sev_score,
        "severity_stage": stage,
        "advice": rec
    }


In [ ]:
def gradio_predict(img: Image.Image):
    out = predict_image(img)
    rec = out["advice"]

    if out["label"] in ["Invalid Input", "Uncertain Prediction", "Other_Crop"]:
        text = (
            f"🩺 Disease: {out['label']}\n"
            f"📊 Confidence: {out['confidence']*100:.2f}%\n"
            f"🌡️ Severity: {out['severity_stage']} (score {out['severity_score']:.2f})\n\n"
            f"📝 What to do:\n{rec.get('summary','')}\n"
            + "\n".join([f"• {a}" for a in rec.get('actions', [])])
        )
    else:
        text = (
            f"🩺 Disease: {out['label']}\n"
            f"📊 Confidence: {out['confidence']*100:.2f}%\n"
            f"🌡️ Severity: {out['severity_stage']} (score {out['severity_score']:.2f})\n\n"
            f"📝 What to do: {rec.get('summary','')}\n"
            + "\n".join([f"• {a}" for a in rec.get('actions', [])]) +
            "\n\n🛡️ Prevention:\n" + "\n".join([f"• {p}" for p in rec.get('prevention', [])])
        )

    return {out['label']: out['confidence']}, text

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type="pil", label="Upload leaf image"),
    outputs=[gr.Label(num_top_classes=len(CLASSES)), gr.Markdown()],
    title="🌱 Crop Disease & Severity Assistant (ResNet-50)",
    description="Upload a leaf photo (Tomato/Potato/Pepper/Rice/Maize). If another crop is uploaded, it will warn with 'Other Crop'."
)

demo.launch(share=True, debug=False)